# RobinReal — Prototype + Debug Dashboard

**Interface:** `search(query: str) -> SearchResult` carrying the ranked listings *and* a per-stage debug trace.

```
query ──► extract_hard_facts ──► hard filter ──► extract_soft_facts ──► filter_soft_facts ──► rank_listings ──► results
             │                      │                  │                       │                    │
             └── trace ─────────────┴── trace ─────────┴── trace ──────────────┴── trace ───────────┴── trace ──► dashboard
```

The four participant functions (`extract_hard_facts`, `extract_soft_facts`, `filter_soft_facts`, `rank_listings`) **match the harness signatures** in [`challenge harness/app/participant/`](../../challenge%20harness/app/participant/) 1:1. Develop here, then copy the function body into the matching harness file — no interface translation.

## 1. Setup

In [5]:
%pip install -q folium pandas

Note: you may need to restart the kernel to use updated packages.


DEPRECATION: Loading egg at c:\users\shaun\miniconda3\lib\site-packages\tab_cpp-1.0-py3.11-win-amd64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at c:\users\shaun\miniconda3\lib\site-packages\ternaryllm-1.0-py3.11-win-amd64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330


## 2. Schemas

Plain dataclasses mirroring `challenge harness/app/models/schemas.py`. When copying code into the harness, swap these for the pydantic imports.

In [6]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Literal

@dataclass
class HardFilters:
    city: list[str] | None = None
    postal_code: list[str] | None = None
    canton: str | None = None
    min_price: int | None = None
    max_price: int | None = None
    min_rooms: float | None = None
    max_rooms: float | None = None
    latitude: float | None = None
    longitude: float | None = None
    radius_km: float | None = None
    features: list[str] | None = None
    offer_type: str | None = None
    object_category: list[str] | None = None
    limit: int = 25
    offset: int = 0
    sort_by: Literal['price_asc', 'price_desc', 'rooms_asc', 'rooms_desc'] | None = None

@dataclass
class ListingData:
    id: str
    title: str
    description: str | None = None
    street: str | None = None
    city: str | None = None
    postal_code: str | None = None
    canton: str | None = None
    latitude: float | None = None
    longitude: float | None = None
    price_chf: int | None = None
    rooms: float | None = None
    living_area_sqm: int | None = None
    available_from: str | None = None
    image_urls: list[str] | None = None
    hero_image_url: str | None = None
    original_listing_url: str | None = None
    features: list[str] = field(default_factory=list)
    offer_type: str | None = None
    object_category: str | None = None
    object_type: str | None = None

@dataclass
class RankedListingResult:
    listing_id: str
    score: float
    reason: str
    listing: ListingData

# Notebook-only extras for debug dashboard.
@dataclass
class StageTrace:
    name: str
    ms: float
    info: dict[str, Any] = field(default_factory=dict)

@dataclass
class SearchResult:
    query: str
    hard_filters: HardFilters
    soft_facts: dict[str, Any]
    candidates: list[dict[str, Any]]        # after hard filter, before soft filter
    soft_filtered: list[dict[str, Any]]     # after soft filter
    ranked: list[RankedListingResult]       # final output (matches harness ListingsResponse.listings)
    traces: list[StageTrace] = field(default_factory=list)

## 3. Pseudo pipeline (stubs)

The four `extract_*`, `filter_soft_facts`, `rank_listings` functions below are the **only** pieces teammates need to implement. Signatures match harness exactly — replace the bodies with real logic.

In [7]:
# ---------------------------------------------------------------------------
# Real Backend API Client
# Replaces PSEUDO_POOL + _hard_filter_pool with live calls to http://localhost:8000
# ---------------------------------------------------------------------------
import httpx

BACKEND_URL = "http://localhost:8000"

def _check_backend():
    try:
        r = httpx.get(f"{BACKEND_URL}/health", timeout=3)
        r.raise_for_status()
        print(f"✅ Backend healthy: {r.json()}")
    except Exception as e:
        print(f"❌ Backend not reachable at {BACKEND_URL}: {e}")

_check_backend()


def _hard_filter_pool(hf) -> list[dict]:
    """
    Calls POST /listings/search/filter with the HardFilters object.
    Returns a list of listing dicts compatible with the rest of the pipeline.
    """
    payload = {}
    if hf.city:           payload["city"] = hf.city
    if hf.postal_code:    payload["postal_code"] = hf.postal_code
    if hf.canton:         payload["canton"] = hf.canton
    if hf.min_price is not None: payload["min_price"] = hf.min_price
    if hf.max_price is not None: payload["max_price"] = hf.max_price
    if hf.min_rooms is not None: payload["min_rooms"] = hf.min_rooms
    if hf.max_rooms is not None: payload["max_rooms"] = hf.max_rooms
    if hf.features:       payload["features"] = hf.features
    if hf.offer_type:     payload["offer_type"] = hf.offer_type
    if hf.object_category: payload["object_category"] = hf.object_category
    payload["limit"]  = hf.limit
    payload["offset"] = hf.offset

    resp = httpx.post(
        f"{BACKEND_URL}/listings/search/filter",
        json={"hard_filters": payload},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()

    # Unwrap RankedListingResult -> flat dict for downstream pipeline stages
    results = []
    for item in data.get("listings", []):
        lst = item.get("listing", {})
        results.append({
            "listing_id":      lst.get("id"),
            "title":           lst.get("title", ""),
            "city":            lst.get("city"),
            "postal_code":     lst.get("postal_code"),
            "canton":          lst.get("canton"),
            "price":           lst.get("price_chf"),
            "rooms":           lst.get("rooms"),
            "latitude":        lst.get("latitude"),
            "longitude":       lst.get("longitude"),
            "description":     lst.get("description", ""),
            "features":        lst.get("features") or [],
            "living_area_sqm": lst.get("living_area_sqm"),
            "available_from":  lst.get("available_from"),
            "image_urls":      lst.get("image_urls") or [],
            "hero_image_url":  lst.get("hero_image_url"),
            "original_listing_url": lst.get("original_listing_url"),
            "offer_type":      lst.get("offer_type"),
            "object_category": lst.get("object_category"),
        })
    return results


def _get_pool_size() -> int:
    """Fetch total listing count from the backend for display in dashboard."""
    try:
        resp = httpx.post(
            f"{BACKEND_URL}/listings/search/filter",
            json={"hard_filters": {"limit": 1, "offset": 0}},
            timeout=10,
        )
        resp.raise_for_status()
        meta = resp.json().get("meta", {})
        return meta.get("total", "?")
    except Exception:
        return "?"

POOL_SIZE = _get_pool_size()
print(f"📦 Backend pool size: {POOL_SIZE} listings")


❌ Backend not reachable at http://localhost:8000: [WinError 10061] No connection could be made because the target machine actively refused it
📦 Backend pool size: ? listings


## 4. `search()` — wires stages + records trace

In [8]:
# ---------------------------------------------------------------------------
# Participant functions (stubs — replace with real implementations)
# ---------------------------------------------------------------------------
import sys, os
# Ensure the project root is on sys.path so `app` is importable
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from app.participant.hard_fact_extraction import extract_hard_facts
from app.participant.soft_fact_extraction import extract_soft_facts
from app.participant.soft_filtering import filter_soft_facts
from app.participant.ranking import rank_listings


# ---------------------------------------------------------------------------
# _timed: thin wrapper that returns (result, elapsed_ms)
# ---------------------------------------------------------------------------
import time

def _timed(fn, *args, **kwargs):
    """Call fn(*args, **kwargs), return (result, elapsed_ms)."""
    t0 = time.perf_counter()
    result = fn(*args, **kwargs)
    ms = (time.perf_counter() - t0) * 1000
    return result, ms


ModuleNotFoundError: No module named 'openai'

In [ ]:
def search(query: str) -> SearchResult:
    """Mirror of the harness search_service: runs the five pipeline stages and records timings."""
    traces: list[StageTrace] = []

    hf, ms = _timed(extract_hard_facts, query)
    traces.append(StageTrace('extract_hard_facts', ms, {'filters': {k: v for k, v in hf.__dict__.items() if v not in (None, [], 25, 0)}}))

    candidates, ms = _timed(_hard_filter_pool, hf)
    traces.append(StageTrace('hard_filter', ms, {'pool_size': POOL_SIZE, 'candidates': len(candidates)}))

    sf, ms = _timed(extract_soft_facts, query)
    traces.append(StageTrace('extract_soft_facts', ms, {'preferences': sf.get('preferences', {})}))

    soft_filtered, ms = _timed(filter_soft_facts, candidates, sf)
    traces.append(StageTrace('filter_soft_facts', ms, {'survivors': len(soft_filtered)}))

    ranked, ms = _timed(rank_listings, soft_filtered, sf)
    traces.append(StageTrace('rank_listings', ms, {'ranked': len(ranked)}))

    return SearchResult(
        query=query, hard_filters=hf, soft_facts=sf,
        candidates=candidates, soft_filtered=soft_filtered, ranked=ranked,
        traces=traces,
    )

## 5. Map view — price pins

Airbnb-style price pills rendered at each listing's lat/lon. Click a pin for title, rooms, score, and reason.

In [ ]:
import folium
from folium.features import DivIcon

PIN_CSS = (
    "display:inline-block;padding:4px 10px;border-radius:16px;"
    "background:#fff;border:1.5px solid #222;box-shadow:0 1px 4px rgba(0,0,0,.25);"
    "font:600 12px/1 -apple-system,BlinkMacSystemFont,Segoe UI,Roboto,sans-serif;"
    "color:#222;white-space:nowrap;"
)
PIN_CSS_TOP = PIN_CSS.replace('#fff', '#111').replace('color:#222', 'color:#fff').replace('#222', '#111')

def _popup_html(r: RankedListingResult, rank: int) -> str:
    l = r.listing
    img = f'<img src="{l.hero_image_url}" style="width:100%;border-radius:6px;margin-bottom:6px">' if l.hero_image_url else ''
    price = f'CHF {l.price_chf:,.0f}/mo' if l.price_chf else 'price n/a'
    rooms = f'{l.rooms:g} rooms' if l.rooms else ''
    return (
        f'<div style="font:13px/1.4 -apple-system,BlinkMacSystemFont,sans-serif;width:220px">'
        f'{img}'
        f'<div style="font-weight:700;margin-bottom:2px">#{rank} · {l.title}</div>'
        f'<div style="color:#555">{l.city or ""} · {rooms}</div>'
        f'<div style="margin-top:4px"><b>{price}</b> · score {r.score:.2f}</div>'
        f'<div style="color:#666;margin-top:4px;font-size:12px">{r.reason}</div>'
        f'</div>'
    )

def show_map(result: SearchResult, *, highlight_top: int = 1):
    pts = [r for r in result.ranked if r.listing.latitude is not None and r.listing.longitude is not None]
    if not pts:
        print('(no coordinates to plot)'); return None
    lat0 = sum(p.listing.latitude for p in pts) / len(pts)
    lon0 = sum(p.listing.longitude for p in pts) / len(pts)
    m = folium.Map(location=[lat0, lon0], zoom_start=12, tiles='CartoDB positron')
    for i, r in enumerate(pts):
        style = PIN_CSS_TOP if i < highlight_top else PIN_CSS
        price = r.listing.price_chf or 0
        html = f'<div style="{style}">CHF {price:,.0f}</div>'
        folium.Marker(
            [r.listing.latitude, r.listing.longitude],
            icon=DivIcon(icon_size=(90, 24), icon_anchor=(45, 12), html=html),
            popup=folium.Popup(_popup_html(r, i + 1), max_width=260),
            tooltip=f'#{i + 1} · {r.listing.title}',
        ).add_to(m)
    if len(pts) > 1:
        m.fit_bounds(
            [[min(p.listing.latitude for p in pts), min(p.listing.longitude for p in pts)],
             [max(p.listing.latitude for p in pts), max(p.listing.longitude for p in pts)]],
            padding=(30, 30),
        )
    return m

## 6. Debug dashboard

In [ ]:
from IPython.display import display, Markdown, HTML
import pandas as pd

# ---------------------------------------------------------------------------
# Parser-output renderer — two visually distinct tables (hard vs soft).
# ---------------------------------------------------------------------------
# Schema (from eval/query_parser/example_results.jsonl):
#   parsed = {"original_query": str,
#             "constraints": [{"source_phrase", "key", "predefined",
#                              "constraint_type" ("hard"|"soft"),
#                              "expression"}, ...]}

def _constraints_table(constraints: list[dict], *, header_color: str, row_bg: str) -> str:
    if not constraints:
        return '<div style="padding:8px 10px;color:#aaa;font:13px sans-serif">none</div>'
    head = (
        f'<thead><tr style="background:{header_color};color:#fff;text-align:left">'
        '<th style="padding:6px 10px">phrase</th>'
        '<th style="padding:6px 10px">key</th>'
        '<th style="padding:6px 10px">expression</th>'
        '<th style="padding:6px 10px;text-align:center">vocab</th>'
        '</tr></thead>'
    )
    rows = ''
    for c in constraints:
        pred = '✓' if c.get('predefined') else '<span style="color:#aaa">free</span>'
        rows += (
            f'<tr style="background:{row_bg}">'
            f'<td style="padding:4px 10px;font-style:italic;color:#555">"{c.get("source_phrase","")}"</td>'
            f'<td style="padding:4px 10px;font:600 13px monospace;color:#222">{c.get("key","")}</td>'
            f'<td style="padding:4px 10px;font:13px monospace;color:#0a3">{c.get("expression","")}</td>'
            f'<td style="padding:4px 10px;text-align:center">{pred}</td>'
            '</tr>'
        )
    return (
        f'<table style="border-collapse:collapse;font:13px sans-serif;width:100%;'
        f'border:1px solid {header_color};border-radius:4px;overflow:hidden">'
        f'{head}<tbody>{rows}</tbody></table>'
    )

def show_parsed_constraints(parsed: dict) -> None:
    constraints = (parsed or {}).get('constraints', [])
    hard  = [c for c in constraints if c.get('constraint_type') == 'hard']
    soft  = [c for c in constraints if c.get('constraint_type') == 'soft']
    other = [c for c in constraints if c.get('constraint_type') not in {'hard', 'soft'}]

    display(HTML(
        f'<div style="font:600 14px sans-serif;color:#c0392b;margin:4px 0 6px">'
        f'🔴 HARD filters — {len(hard)} (must satisfy)</div>'
        + _constraints_table(hard, header_color='#c0392b', row_bg='#fff5f5')
    ))
    display(HTML(
        f'<div style="font:600 14px sans-serif;color:#2980b9;margin:14px 0 6px">'
        f'🔵 SOFT preferences — {len(soft)} (boost ranking)</div>'
        + _constraints_table(soft, header_color='#2980b9', row_bg='#f0f7ff')
    ))
    if other:
        display(HTML(
            f'<div style="font:600 14px sans-serif;color:#888;margin:14px 0 6px">'
            f'⚠ Unclassified — {len(other)}</div>'
            + _constraints_table(other, header_color='#888', row_bg='#fafafa')
        ))


# ---------------------------------------------------------------------------
# Fallback: pretty tables derived from HardFilters + soft_facts dict when
# the pipeline hasn't attached parser output yet.
# ---------------------------------------------------------------------------

def _legacy_hard_soft_as_constraints(hf: HardFilters, soft_facts: dict) -> list[dict]:
    """Lift legacy HardFilters + soft prefs into constraint-shaped rows so
    the same renderer can display both pipeline modes."""
    out: list[dict] = []
    hf_expr = {
        'city':          lambda v: f"this in {v}",
        'postal_code':   lambda v: f"this in {v}",
        'canton':        lambda v: f"this == {v!r}",
        'min_price':     lambda v: f"this >= {v}",
        'max_price':     lambda v: f"this <= {v}",
        'min_rooms':     lambda v: f"this >= {v}",
        'max_rooms':     lambda v: f"this <= {v}",
        'features':      lambda v: f"all_in {v}",
        'offer_type':    lambda v: f"this == {v!r}",
        'object_category': lambda v: f"this in {v}",
    }
    for k, fmt in hf_expr.items():
        v = getattr(hf, k, None)
        if v in (None, [], ''): continue
        out.append({
            'source_phrase': '—', 'key': k, 'predefined': True,
            'constraint_type': 'hard', 'expression': fmt(v),
        })
    for k, v in (soft_facts or {}).get('preferences', {}).items():
        out.append({
            'source_phrase': '—', 'key': k, 'predefined': False,
            'constraint_type': 'soft', 'expression': f"weight={v}",
        })
    return out


# ---------------------------------------------------------------------------
# Ranked-results table — compact, highlights the top result.
# ---------------------------------------------------------------------------

def _ranked_html(ranked: list[RankedListingResult], *, top_n: int = 10) -> HTML:
    if not ranked:
        return HTML('<div style="color:#aaa;font:13px sans-serif">(nothing ranked)</div>')
    head = (
        '<thead><tr style="background:#222;color:#fff;text-align:left">'
        '<th style="padding:6px 10px;text-align:right">#</th>'
        '<th style="padding:6px 10px">listing</th>'
        '<th style="padding:6px 10px">city</th>'
        '<th style="padding:6px 10px;text-align:right">CHF</th>'
        '<th style="padding:6px 10px;text-align:right">rooms</th>'
        '<th style="padding:6px 10px;text-align:right">score</th>'
        '<th style="padding:6px 10px">why</th>'
        '</tr></thead>'
    )
    rows = ''
    for i, r in enumerate(ranked[:top_n], 1):
        l = r.listing
        bg = '#fffbe6' if i == 1 else ('#fff' if i % 2 else '#fafafa')
        price = f'{l.price_chf:,}' if l.price_chf else '—'
        rooms = f'{l.rooms:g}' if l.rooms else '—'
        rows += (
            f'<tr style="background:{bg}">'
            f'<td style="padding:6px 10px;text-align:right;color:#888;font-variant-numeric:tabular-nums">{i}</td>'
            f'<td style="padding:6px 10px;font-weight:600;color:#222">{l.title}</td>'
            f'<td style="padding:6px 10px;color:#555">{l.city or ""}</td>'
            f'<td style="padding:6px 10px;text-align:right;font-variant-numeric:tabular-nums">{price}</td>'
            f'<td style="padding:6px 10px;text-align:right;font-variant-numeric:tabular-nums">{rooms}</td>'
            f'<td style="padding:6px 10px;text-align:right;font:600 13px monospace;color:#0a3">{r.score:.2f}</td>'
            f'<td style="padding:6px 10px;color:#555;font-size:12px">{r.reason}</td>'
            '</tr>'
        )
    return HTML(
        f'<table style="border-collapse:collapse;font:13px sans-serif;width:100%">{head}<tbody>{rows}</tbody></table>'
    )


# ---------------------------------------------------------------------------
# dashboard() — 3 sections: filters, candidate funnel, ranked results
# ---------------------------------------------------------------------------

def dashboard(result: SearchResult) -> None:
    r = result
    display(Markdown(f'# 🔍 Query\n> {r.query}'))

    # 1. Hard/soft filter tables
    display(Markdown('## 1 · Extracted filters'))
    parsed = r.soft_facts.get('parsed') if isinstance(r.soft_facts, dict) else None
    if parsed:
        show_parsed_constraints(parsed)
    else:
        # Fall back to constraint-shaped rendering of the legacy output,
        # so the visual stays consistent either way.
        show_parsed_constraints({'constraints': _legacy_hard_soft_as_constraints(r.hard_filters, r.soft_facts)})

    # 2. Candidate funnel
    n_pool = POOL_SIZE
    n_hard = len(r.candidates)
    n_soft = len(r.soft_filtered)
    display(Markdown(
        f'## 2 · Candidate funnel\n'
        f'**{n_pool}** in pool → **{n_hard}** after hard filter → **{n_soft}** after soft filter'
    ))

    # 3. Ranked results
    display(Markdown(f'## 3 · Ranked results — top {min(10, len(r.ranked))} of {len(r.ranked)}'))
    display(_ranked_html(r.ranked, top_n=10))


def show_results(result: SearchResult) -> None:
    """Lean output: ranked table + map (no debug)."""
    display(Markdown(f'### Query\n> {result.query}'))
    if result.ranked:
        display(_ranked_html(result.ranked, top_n=10))
        m = show_map(result)
        if m is not None: display(m)
    else:
        display(Markdown('_No listings matched._'))

## 7. Demo — lean output with map

In [ ]:
show_results(search('3.5-room bright apartment in Zurich under 2800'))

## 8. Debug mode — full dashboard

In [ ]:
dashboard(search('3.5-room bright apartment in Zurich under 2800'))

In [ ]:
dashboard(search('modern quiet studio in Geneva'))

In [ ]:
dashboard(search('family friendly flat in Winterthur'))